# CNN Project: Classifying Dogs vs Cats (Color Images)

## What are we doing?
We are building a **Convolutional Neural Network (CNN)** that looks at **real color photos** of dogs and cats and learns to tell them apart.

This project uses the same core ideas as the [CNN_Mnist.ipynb](./CNN_Mnist.ipynb) (LeNet-5) notebook — Convolution + Pooling + Dense layers — but adapted for a harder, more realistic problem.

## How is this different from the MNIST digit project?

| | MNIST digits (LeNet-5) | Dogs vs Cats (this project) |
|---|---|---|
| Image size | 28 x 28 pixels (tiny) | 160 x 160 pixels (bigger, real photos) |
| Color channels | 1 (grayscale) | 3 (Red, Green, Blue — color!) |
| Number of classes | 10 (digits 0-9) | 2 (dog, cat) |
| Output layer | `Dense(10, softmax)` | `Dense(1, sigmoid)` |
| Hidden activation | `tanh` (old-style) | `relu` (modern standard) |
| Pooling | `AveragePooling2D` | `MaxPooling2D` (modern standard) |
| Overfitting risk | Low (simple images) | Higher (real photos vary a lot) → we add **Dropout** |

## The Big Picture

```
Color Photos  →  Resize & Normalize  →  Labels (cat=0, dog=1)
     ↓
Conv2D + MaxPool (detect edges/colors)
     ↓
Conv2D + MaxPool (detect fur, shapes)
     ↓
Conv2D + MaxPool (detect ears, eyes, faces)
     ↓
Flatten (make it a flat list)
     ↓
Dense layer (combine clues) + Dropout (avoid memorizing)
     ↓
Output layer (probability: dog or cat?)
     ↓
Compile (set learning rules) → Train (practice) → Evaluate (final exam)
```

## Step 1: Install and Import the Libraries

- `tensorflow` → the main deep learning library.
- `tensorflow_datasets` → gives us easy access to ready-made datasets, including **"cats_vs_dogs"**, without needing to manually download and unzip files.
- `layers`, `models` → building blocks to construct our CNN.
- `matplotlib` → to visualize images and training progress.

The `pip install` line makes sure `tensorflow_datasets` is available (only needs to run once).

In [ ]:
# Install tensorflow_datasets if it's not already available
%pip install -q tensorflow_datasets

import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers, models

print("TensorFlow version:", tf.__version__)

## Step 2: Load the Dogs vs Cats Dataset

`tensorflow_datasets` (`tfds`) downloads the **"cats_vs_dogs"** dataset for us automatically (about 25,000 real color photos of cats and dogs).

This dataset only comes with one big `train` split, so we manually slice it ourselves:
- **First 80%** → `train_ds` (used to teach the model)
- **Last 20%** → `val_ds` (used to check progress while training, on images not trained on)

`as_supervised=True` gives us clean `(image, label)` pairs, and `with_info=True` gives us extra details like class names.

In [ ]:
# Load the dataset: 80% for training, 20% for validation
(train_ds, val_ds), dataset_info = tfds.load(
    'cats_vs_dogs',
    split=['train[:80%]', 'train[80%:]'],
    with_info=True,
    as_supervised=True
)

class_names = dataset_info.features['label'].names
print("Class names:", class_names)
print("Number of training examples:", dataset_info.splits['train'].num_examples)

## Step 3: Look at Some Sample Images

Before building the model, it's always a good idea to **actually look at the data**. This helps us understand what the model will be learning from.

We grab one small batch of 9 images from `train_ds` and display them with their correct labels (cat or dog).

In [ ]:
# Display 9 sample images with their labels
plt.figure(figsize=(8, 8))

for i, (image, label) in enumerate(train_ds.take(9)):
    plt.subplot(3, 3, i + 1)
    plt.imshow(image.numpy().astype("uint8"))
    plt.title(class_names[label.numpy()])
    plt.axis('off')

plt.tight_layout()
plt.show()

## Step 4: Preprocess the Images (Resize, Normalize, Batch)

Real photos come in **all kinds of sizes**, but a neural network needs every input to be **exactly the same shape**. So we:

1. **Resize** every image to a fixed size: `160 x 160` pixels (still color, so shape becomes `160 x 160 x 3`).
2. **Normalize** pixel values from the `0-255` range down to `0-1`, just like we did for MNIST.
3. **Batch** the images — group them into batches of 32, so the model trains on 32 images at a time.
4. **Shuffle** the training data so the model doesn't learn the order of the images.
5. **Prefetch** — lets the computer prepare the next batch while the current one is training, which speeds things up.

> Note: We shuffle and augment only the **training** set. The **validation** set is kept as-is (no shuffling) since we only use it to check performance, not to learn.

In [ ]:
IMG_SIZE = 160
BATCH_SIZE = 32

def preprocess(image, label):
    # Resize every image to the same 160x160 size
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    # Normalize pixel values from [0, 255] to [0, 1]
    image = image / 255.0
    return image, label

train_ds = (
    train_ds
    .map(preprocess)
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    val_ds
    .map(preprocess)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print("Data pipeline ready!")

## Step 5: Build the CNN Model

This CNN follows the **same pattern** as LeNet-5 (Conv + Pool blocks, then Flatten, then Dense layers), but modernized for harder, real-world color photos.

| Layer | What it does (simple terms) |
|---|---|
| `Conv2D(filters=32, kernel_size=(3,3), activation='relu')` | Scans the image with **32 small 3x3 filters** to detect basic patterns: edges, colors, simple textures. |
| `MaxPooling2D(2,2)` | Shrinks the image by keeping only the **strongest (maximum) signal** in each 2x2 block — like zooming out while keeping the most important details. |
| `Conv2D(filters=64, kernel_size=(3,3), activation='relu')` | A deeper scan with **64 filters** to detect more complex patterns, like fur texture or ear shapes. |
| `MaxPooling2D(2,2)` | Shrinks the image again. |
| `Conv2D(filters=128, kernel_size=(3,3), activation='relu')` | An even deeper scan with **128 filters** to detect high-level features like faces, eyes, or body shape. |
| `MaxPooling2D(2,2)` | Shrinks the image one more time. |
| `Flatten()` | Turns the final 2D grid of detected features into **one flat list of numbers**. |
| `Dense(units=128, activation='relu')` | A fully-connected layer that combines all the detected clues together. |
| `Dropout(0.5)` | **Randomly switches off 50% of neurons** during training. This forces the model to not rely too heavily on any single neuron, which helps prevent **overfitting** (memorizing training photos instead of learning general patterns). This is new compared to LeNet-5, because real photos are much more prone to overfitting than simple digit images. |
| `Dense(units=1, activation='sigmoid')` | The **output layer** — just 1 neuron, since this is **binary classification** (dog vs cat). `sigmoid` gives a probability between 0 and 1: closer to 0 means "cat", closer to 1 means "dog". |

**Why `relu` + `MaxPooling2D` instead of `tanh` + `AveragePooling2D` (like LeNet-5)?** These are the modern, more commonly-used choices for today's CNNs — `relu` trains faster and `MaxPooling2D` tends to keep the most noticeable features, which works better for busy, detailed real-world photos.

In [ ]:
model = models.Sequential([
    # Block 1: Conv2D + MaxPooling
    layers.Conv2D(filters=32, kernel_size=(3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.MaxPooling2D(pool_size=(2, 2)),

    # Block 2: Conv2D + MaxPooling
    layers.Conv2D(filters=64, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),

    # Block 3: Conv2D + MaxPooling
    layers.Conv2D(filters=128, kernel_size=(3, 3), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),

    # Flatten + Fully Connected (Dense) layers
    layers.Flatten(),
    layers.Dense(units=128, activation='relu'),
    layers.Dropout(0.5),

    # Output Layer: 1 neuron, Sigmoid activation (binary: dog or cat)
    layers.Dense(units=1, activation='sigmoid')
])

model.summary()

## Step 6: Compile the Model

Before training, we tell the model **how to learn**:

| Setting | What it does | Our choice |
|---|---|---|
| **Optimizer** | How the model updates its weights | `adam` (adaptive, good general-purpose choice) |
| **Loss Function** | How to measure error | `binary_crossentropy` (because there are only 2 classes: dog/cat) |
| **Metrics** | What to track during training | `accuracy` |

We use `binary_crossentropy` here (not `categorical_crossentropy` like in the MNIST project) because we only have **2 classes**, and our output layer is a single `sigmoid` neuron instead of a 10-neuron `softmax`.

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Model compiled successfully!")

## Step 7: Train the Model

Now we **fit** the model using our training data, and check progress on the validation data after every epoch.

- **epochs=10** → the model studies the entire training dataset 10 times.
- **validation_data=val_ds** → after each epoch, the model checks itself on images it did NOT train on, so we get an honest sense of how well it's learning.

> Training on real color photos takes longer than on tiny MNIST digits — this step may take a while depending on your machine (GPU recommended).

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

print("\nTraining complete!")

## Step 8: Visualize Training Progress

Let's plot how the model improved over each epoch.
- **Loss** should go DOWN (fewer mistakes)
- **Accuracy** should go UP (better predictions)

If **validation loss** starts going UP while **training loss** keeps going down, that's a sign of **overfitting** — the model is starting to memorize the training photos instead of learning general patterns.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: Loss ---
axes[0].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_title('Model Loss Over Epochs', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# --- Plot 2: Accuracy ---
axes[1].plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[1].set_title('Model Accuracy Over Epochs', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 9: Evaluate the Model

This checks the model's final performance on the **validation set** — images it never directly trained on — giving us an honest measure of how good it really is.

In [ ]:
val_loss, val_acc = model.evaluate(val_ds)
print(f"\nValidation accuracy: {val_acc:.4f}")

## Step 10: Make a Prediction on a New Image

Let's grab one image from the validation set and see what the model predicts.

Remember: the output is a **probability between 0 and 1**.
- Closer to `0` → model thinks it's a **cat**
- Closer to `1` → model thinks it's a **dog**

We use `0.5` as the cutoff point to decide the final answer.

In [ ]:
# Take a single batch and predict on the first image
for images, labels in val_ds.take(1):
    sample_image = images[0]
    true_label = labels[0].numpy()
    break

probability = model.predict(tf.expand_dims(sample_image, axis=0))[0][0]
predicted_label = 1 if probability >= 0.5 else 0

plt.imshow(sample_image.numpy())
plt.axis('off')
plt.title(
    f"True: {class_names[true_label]} | "
    f"Predicted: {class_names[predicted_label]} ({probability:.2f})"
)
plt.show()